In [8]:
import numpy as np
import pandas as pd
from mixedimputer import TimeSeriesImputer
from mixedimputer.corrupt_data import DataCorrupter


In [9]:
# ---------------------------------------------------------------------------
# 1. Create a synthetic 100-second vehicle telemetry dataset
# ---------------------------------------------------------------------------

np.random.seed(42)
n = 100

# Use numeric time (seconds) — MixedImputer requires all columns to be
# either numeric or categorical; datetime64 is neither.
time = np.arange(n, dtype=float)  # 0, 1, 2, ..., 99 seconds

# Speed: sinusoidal cruise with noise (auto-correlated via cumsum of small steps)
speed_base = 50 + 15 * np.sin(np.linspace(0, 3 * np.pi, n))  # oscillate 35–65 km/h
speed = speed_base + np.random.normal(0, 2, n).cumsum() * 0.05  # slight drift
speed = np.clip(speed, 0, 130)

# Engine temperature: gradually warming up, periodic cooling, + noise
temp_base = 80 + 10 * np.sin(np.linspace(0, 2 * np.pi, n))  # oscillate 70–90 °C
temp = temp_base + np.random.normal(0, 1, n).cumsum() * 0.02  # slight drift
temp = np.clip(temp, 60, 110)

df_original = pd.DataFrame({
    "timestamp": time,
    "speed_kmh": speed,
    "engine_temp_c": temp,
})

df_original.head(10)


,timestamp,speed_kmh,engine_temp_c
0,0.0,50.049671,79.971693
1,1.0,51.461686,80.597519
2,2.0,52.939383,81.222350
3,3.0,54.478905,81.832892
4,4.0,55.804438,82.448634
5,5.0,57.079486,83.065570
6,6.0,58.473621,83.699583
7,7.0,59.713137,84.284400
8,8.0,60.744990,84.853569
9,9.0,61.784305,85.396521


In [10]:
# ---------------------------------------------------------------------------
# 2. Corrupt ~5% of values in each numeric column using DataCorrupter (MCAR)
# ---------------------------------------------------------------------------

corrupter = DataCorrupter(
    mechanism="MCAR",
    corruption_fraction=0.15,            # 15% per column
    numeric_columns=["speed_kmh", "engine_temp_c"],  # target these two columns
    random_state=42,
)

df_corrupted, mask, df_original_copy = corrupter.corrupt(df_original)

# Show corruption stats
total_cells = mask[["speed_kmh", "engine_temp_c"]].size
corrupted_cells = mask[["speed_kmh", "engine_temp_c"]].sum().sum()
print(f"Total cells (speed + temp): {total_cells}")
print(f"Corrupted cells:              {corrupted_cells}")
print(f"Actual corruption rate:        {corrupted_cells / total_cells:.2%}")
print()
print("First 10 rows — corrupted (NaN where values were removed):")
df_corrupted.head(10)


Total cells (speed + temp): 200
Corrupted cells:              33
Actual corruption rate:        16.50%

First 10 rows — corrupted (NaN where values were removed):


,timestamp,speed_kmh,engine_temp_c
0,0.0,50.049671,NaN
1,1.0,51.461686,80.597519
2,2.0,52.939383,81.222350
3,3.0,54.478905,81.832892
4,4.0,55.804438,82.448634
5,5.0,57.079486,83.065570
6,6.0,NaN,83.699583
7,7.0,59.713137,84.284400
8,8.0,60.744990,84.853569
9,9.0,61.784305,NaN


In [11]:
# ---------------------------------------------------------------------------
# 3. Impute with TimeSeriesImputer
# ---------------------------------------------------------------------------

imputer = TimeSeriesImputer(
    time_column="timestamp",
    lags=[1, 2, 3],                     # use t-1, t-2, t-3 as predictors
    rolling_windows=[5, 10],            # 5s and 10s rolling means
    initial_strategy="forward_fill",     # time-aware initial fill
    max_iter=10,
    random_state=42,
    verbose=0,
)

df_imputed = imputer.fit_transform(df_corrupted)

print("Imputation complete.")
df_imputed.head(10)


Imputation complete.


,timestamp,speed_kmh,engine_temp_c
0,0.0,50.049671,80.954740
1,1.0,51.461686,80.597519
2,2.0,52.939383,81.222350
3,3.0,54.478905,81.832892
4,4.0,55.804438,82.448634
5,5.0,57.079486,83.065570
6,6.0,58.242487,83.699583
7,7.0,59.713137,84.284400
8,8.0,60.744990,84.853569
9,9.0,61.784305,85.429491


In [12]:
# ---------------------------------------------------------------------------
# 4. Direct comparison: original vs imputed at the corrupted positions
# ---------------------------------------------------------------------------

comparisons = []

for col in ["speed_kmh", "engine_temp_c"]:
    corrupted_idx = mask.index[mask[col]]  # rows where this column was corrupted
    for idx in corrupted_idx:
        comparisons.append({
            "column": col,
            "timestamp": df_original.loc[idx, "timestamp"],
            "original": df_original.loc[idx, col],
            "imputed": df_imputed.loc[idx, col],
            "absolute_error": abs(df_original.loc[idx, col] - df_imputed.loc[idx, col]),
        })

df_comp = pd.DataFrame(comparisons)

# --- Summary statistics ---
print("=" * 70)
print("DIRECT COMPARISON — ORIGINAL vs. IMPUTED (corrupted positions only)")
print("=" * 70)

for col in ["speed_kmh", "engine_temp_c"]:
    sub = df_comp[df_comp["column"] == col]
    mae = sub["absolute_error"].mean()
    rmse = (sub["absolute_error"] ** 2).mean() ** 0.5
    n_corrupted = len(sub)
    print(f"\n--- {col} ({n_corrupted} corrupted values) ---")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

# --- Show every corrupted position ---
print("\n" + "=" * 70)
print("ROW-BY-ROW COMPARISON (all corrupted positions)")
print("=" * 70)
display(df_comp)


DIRECT COMPARISON — ORIGINAL vs. IMPUTED (corrupted positions only)

--- speed_kmh (19 corrupted values) ---
  MAE:  0.7411
  RMSE: 0.9804

--- engine_temp_c (14 corrupted values) ---
  MAE:  0.2889
  RMSE: 0.3866

ROW-BY-ROW COMPARISON (all corrupted positions)


,column,timestamp,original,imputed,absolute_error
0,speed_kmh,6.0,58.473621,58.242487,0.231134
1,speed_kmh,10.0,62.620359,63.097225,0.476867
2,speed_kmh,21.0,63.425870,64.313674,0.887804
3,speed_kmh,29.0,55.010496,54.949306,0.061190
4,speed_kmh,32.0,50.985107,50.696507,0.288601
5,speed_kmh,37.0,43.663648,43.048669,0.614979
6,speed_kmh,40.0,39.926915,37.377328,2.549587
7,speed_kmh,42.0,37.868628,37.292202,0.576425
8,speed_kmh,56.0,36.753673,37.231265,0.477592
9,speed_kmh,58.0,38.623332,37.853900,0.769432


In [13]:
# ---------------------------------------------------------------------------
# 5. Interactive visual: overlay original vs imputed (Plotly)
# ---------------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

TEMPLATE = "plotly_white"

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Speed (km/h)", "Engine Temperature (°C)"),
    vertical_spacing=0.08,
)

for row, col in enumerate(["speed_kmh", "engine_temp_c"], start=1):
    corrupt_idx = mask.index[mask[col]]

    # Full original line
    fig.add_trace(
        go.Scatter(
            x=df_original["timestamp"], y=df_original[col],
            mode="lines", name=f"{col} — original",
            line=dict(color="black", width=1), opacity=0.5,
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Corrupted original values (red markers)
    fig.add_trace(
        go.Scatter(
            x=df_original.loc[corrupt_idx, "timestamp"],
            y=df_original.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — original (corrupted)",
            marker=dict(color="red", size=8, symbol="circle"),
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Imputed values (blue X markers)
    fig.add_trace(
        go.Scatter(
            x=df_imputed.loc[corrupt_idx, "timestamp"],
            y=df_imputed.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — imputed",
            marker=dict(color="blue", size=8, symbol="x"),
            legendgroup=col,
        ),
        row=row, col=1,
    )

fig.update_xaxes(title_text="Timestamp (seconds)", row=2, col=1)
fig.update_layout(
    title="TimeSeriesImputer: Original vs. Imputed (red = removed, blue ✕ = imputed)",
    template=TEMPLATE,
    height=600,
    hovermode="x unified",
)
fig.show()


# TimeSeriesImputer — Direct Comparison Benchmark

Create a synthetic 100-second vehicle telemetry dataset, corrupt 5% of values per column, impute with `TimeSeriesImputer`, and compare imputed values against the original ground truth.

# ── GPU Sensor Log Benchmark ──

Load real-world GPU sensor data logged during a gaming session, corrupt two key columns, impute with `TimeSeriesImputer`, and compare against ground truth.

In [14]:
# ---------------------------------------------------------------------------
# 6. Load & prepare GPU sensor log
# ---------------------------------------------------------------------------

# The log file is comma-separated with lots of whitespace padding.
# Use latin1 encoding — the file uses non-UTF-8 characters (e.g. degree symbol).
df_raw = pd.read_csv(
    "../data/gpu_sensor_log.txt",
    skipinitialspace=True,   # strip leading spaces after commas
    encoding="latin1",
)

# Strip whitespace from column names (they have leading/trailing spaces)
df_raw.columns = df_raw.columns.str.strip()

# Parse the Date column to datetime
df_raw["Date"] = pd.to_datetime(df_raw["Date"])

# Convert datetime to numeric seconds-from-start for TimeSeriesImputer
df_raw["time_sec"] = (df_raw["Date"] - df_raw["Date"].iloc[0]).dt.total_seconds().astype(float)

print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
print(f"Time range: {df_raw['Date'].iloc[0]} → {df_raw['Date'].iloc[-1]}")
print(f"Duration: {df_raw['time_sec'].iloc[-1]:.0f} seconds")
print(f"\nColumns: {list(df_raw.columns)}")
df_raw.head()


Loaded 356 rows, 17 columns
Time range: 2026-05-23 11:29:11 → 2026-05-23 11:35:06
Duration: 355 seconds

Columns: ['Date', 'GPU Clock [MHz]', 'Memory Clock [MHz]', 'GPU Temperature [°C]', 'GPU Temperature (Hot Spot) [°C]', 'Fan Speed (%) [%]', 'Fan Speed (RPM) [RPM]', 'GPU Load [%]', 'Memory Controller Load [%]', 'Memory Used (Dedicated) [MB]', 'Memory Used (Dynamic) [MB]', 'GPU Chip Power Draw [W]', 'GPU Voltage [V]', 'CPU Temperature [°C]', 'System Memory Used [MB]', 'Unnamed: 15', 'time_sec']


,Date,GPU Clock [MHz],Memory Clock [MHz],GPU Temperature [°C],GPU Temperature (Hot Spot) [°C],Fan Speed (%) [%],Fan Speed (RPM) [RPM],GPU Load [%],Memory Controller Load [%],Memory Used (Dedicated) [MB],Memory Used (Dynamic) [MB],GPU Chip Power Draw [W],GPU Voltage [V],CPU Temperature [°C],System Memory Used [MB],Unnamed: 15,time_sec
0,2026-05-23 11:29:11,42.0,1988.0,56.0,59.0,0,0,0,0,5250,951,46.0,0.95,47.1,14800,NaN,0.0
1,2026-05-23 11:29:12,96.0,1988.0,55.0,59.0,37,1130,1,0,5258,951,46.0,0.95,47.8,14781,NaN,1.0
2,2026-05-23 11:29:13,98.0,1990.0,56.0,59.0,36,1100,0,0,5258,951,45.0,0.95,48.3,14780,NaN,2.0
3,2026-05-23 11:29:14,20.0,1988.0,55.0,59.0,37,1119,0,0,5248,951,46.0,0.95,48.5,14821,NaN,3.0
4,2026-05-23 11:29:15,207.0,1990.0,55.0,59.0,37,1123,2,0,5248,952,44.0,0.95,49.1,14837,NaN,4.0


In [15]:
# ---------------------------------------------------------------------------
# 7. Select target columns, keep time, and corrupt
# ---------------------------------------------------------------------------

TARGET_COLS = ["GPU Temperature [°C]", "Fan Speed (RPM) [RPM]"]

# Build a clean DataFrame with just the columns we need
df_gpu = df_raw[["time_sec"] + TARGET_COLS].copy()

# Corrupt ~15% of each target column
corrupter_gpu = DataCorrupter(
    mechanism="MCAR",
    corruption_fraction=0.1,
    numeric_columns=TARGET_COLS,
    random_state=42,
)

df_gpu_corrupted, mask_gpu, df_gpu_original = corrupter_gpu.corrupt(df_gpu)

# Show corruption stats
for col in TARGET_COLS:
    n_corrupt = mask_gpu[col].sum()
    n_total = len(df_gpu)
    print(f"{col}: {n_corrupt}/{n_total} corrupted ({n_corrupt/n_total:.1%})")

print(f"\nFirst 10 rows — corrupted:")
df_gpu_corrupted.head(10)


GPU Temperature [°C]: 43/356 corrupted (12.1%)
Fan Speed (RPM) [RPM]: 39/356 corrupted (11.0%)

First 10 rows — corrupted:


,time_sec,GPU Temperature [°C],Fan Speed (RPM) [RPM]
0,0.0,56.0,NaN
1,1.0,55.0,1130.0
2,2.0,56.0,1100.0
3,3.0,55.0,1119.0
4,4.0,55.0,1123.0
5,5.0,55.0,1151.0
6,6.0,NaN,1163.0
7,7.0,55.0,1167.0
8,8.0,54.0,1167.0
9,9.0,54.0,1166.0


In [16]:
# ---------------------------------------------------------------------------
# 8. Impute with TimeSeriesImputer
# ---------------------------------------------------------------------------

imputer_gpu = TimeSeriesImputer(
    time_column="time_sec",
    lags=[1, 2, 3, 5],                  # 1s, 2s, 3s, 5s lags
    rolling_windows=[10, 30],           # 10s and 30s rolling means
    initial_strategy="forward_fill",     # time-aware initial fill
    max_iter=10,
    random_state=42,
    verbose=0,
)

df_gpu_imputed = imputer_gpu.fit_transform(df_gpu_corrupted)

print("Imputation complete.")
df_gpu_imputed.head(10)


Imputation complete.


c:\repositories\mixed-imputer\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,time_sec,GPU Temperature [°C],Fan Speed (RPM) [RPM]
0,0.0,56.000000,1329.890167
1,1.0,55.000000,1130.000000
2,2.0,56.000000,1100.000000
3,3.0,55.000000,1119.000000
4,4.0,55.000000,1123.000000
5,5.0,55.000000,1151.000000
6,6.0,55.033364,1163.000000
7,7.0,55.000000,1167.000000
8,8.0,54.000000,1167.000000
9,9.0,54.000000,1166.000000


In [17]:
# ---------------------------------------------------------------------------
# 9. Direct comparison: original vs imputed
# ---------------------------------------------------------------------------

comparisons_gpu = []

for col in TARGET_COLS:
    corrupted_idx = mask_gpu.index[mask_gpu[col]]
    for idx in corrupted_idx:
        comparisons_gpu.append({
            "column": col,
            "time_sec": df_gpu_original.loc[idx, "time_sec"],
            "original": df_gpu_original.loc[idx, col],
            "imputed": df_gpu_imputed.loc[idx, col],
            "absolute_error": abs(df_gpu_original.loc[idx, col] - df_gpu_imputed.loc[idx, col]),
        })

df_comp_gpu = pd.DataFrame(comparisons_gpu)

# --- Summary statistics ---
print("=" * 70)
print("GPU SENSOR LOG — DIRECT COMPARISON (corrupted positions only)")
print("=" * 70)

for col in TARGET_COLS:
    sub = df_comp_gpu[df_comp_gpu["column"] == col]
    mae = sub["absolute_error"].mean()
    rmse = (sub["absolute_error"] ** 2).mean() ** 0.5
    max_err = sub["absolute_error"].max()
    n_corrupted = len(sub)
    # Also show the scale of the data for context
    col_min = df_gpu_original[col].min()
    col_max = df_gpu_original[col].max()
    col_range = col_max - col_min
    print(f"\n--- {col} ({n_corrupted} corrupted values) ---")
    print(f"  Data range: {col_min:.1f} – {col_max:.1f} (span: {col_range:.1f})")
    print(f"  MAE:  {mae:.4f}  ({mae/col_range*100:.2f}% of range)")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  Max error: {max_err:.4f}")

# --- Show every corrupted position ---
print("\n" + "=" * 70)
print("ROW-BY-ROW COMPARISON (all corrupted positions)")
print("=" * 70)
display(df_comp_gpu)


GPU SENSOR LOG — DIRECT COMPARISON (corrupted positions only)

--- GPU Temperature [°C] (43 corrupted values) ---
  Data range: 54.0 – 73.0 (span: 19.0)
  MAE:  0.6425  (3.38% of range)
  RMSE: 1.1956
  Max error: 5.8966

--- Fan Speed (RPM) [RPM] (39 corrupted values) ---
  Data range: 0.0 – 1861.0 (span: 1861.0)
  MAE:  42.5003  (2.28% of range)
  RMSE: 213.1989
  Max error: 1329.8902

ROW-BY-ROW COMPARISON (all corrupted positions)


,column,time_sec,original,imputed,absolute_error
0,GPU Temperature [°C],6.0,55.0,55.033364,0.033364
1,GPU Temperature [°C],10.0,54.0,54.818368,0.818368
2,GPU Temperature [°C],29.0,66.0,66.229714,0.229714
3,GPU Temperature [°C],32.0,67.0,66.300057,0.699943
4,GPU Temperature [°C],37.0,67.0,66.234719,0.765281
...,...,...,...,...,...
77,Fan Speed (RPM) [RPM],284.0,1089.0,1090.863169,1.863169
78,Fan Speed (RPM) [RPM],294.0,1087.0,1094.216814,7.216814
79,Fan Speed (RPM) [RPM],316.0,1442.0,1452.031194,10.031194
80,Fan Speed (RPM) [RPM],345.0,1094.0,1092.756732,1.243268


In [18]:
# ---------------------------------------------------------------------------
# 10. Interactive visual: GPU data — original vs imputed
# ---------------------------------------------------------------------------

fig_gpu = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("GPU Temperature (°C)", "Fan Speed (RPM)"),
    vertical_spacing=0.08,
)

for row, col in enumerate(TARGET_COLS, start=1):
    corrupt_idx = mask_gpu.index[mask_gpu[col]]

    # Full original line
    fig_gpu.add_trace(
        go.Scatter(
            x=df_gpu_original["time_sec"], y=df_gpu_original[col],
            mode="lines", name=f"{col} — original",
            line=dict(color="black", width=1), opacity=0.5,
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Corrupted original values (red markers)
    fig_gpu.add_trace(
        go.Scatter(
            x=df_gpu_original.loc[corrupt_idx, "time_sec"],
            y=df_gpu_original.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — original (corrupted)",
            marker=dict(color="red", size=6, symbol="circle"),
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Imputed values (blue X markers)
    fig_gpu.add_trace(
        go.Scatter(
            x=df_gpu_imputed.loc[corrupt_idx, "time_sec"],
            y=df_gpu_imputed.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — imputed",
            marker=dict(color="blue", size=8, symbol="x",
                        line=dict(width=1.5)),
            legendgroup=col,
        ),
        row=row, col=1,
    )

fig_gpu.update_xaxes(title_text="Time (seconds since start)", row=2, col=1)
fig_gpu.update_layout(
    title="GPU Sensor Log — TimeSeriesImputer: Original vs. Imputed (red = removed, blue ✕ = imputed)",
    template=TEMPLATE,
    height=600,
    hovermode="x unified",
)
fig_gpu.show()


# ── GPU Sensor Log — Multi-Level Benchmark ──

Evaluate `TimeSeriesImputer` against forward-fill and mean-imputation baselines
at **1%**, **10%**, and **30%** corruption on the two GPU sensor columns.

In [19]:
# ---------------------------------------------------------------------------
# 11. Benchmark function for multi-level comparison
# ---------------------------------------------------------------------------

from sklearn.metrics import mean_squared_error

def run_ts_benchmark(df_clean: pd.DataFrame, fraction: float,
                     target_cols: list, time_col: str = "time_sec",
                     random_state: int = 42) -> dict:
    """Corrupt, impute, and evaluate for a single corruption fraction."""

    # 1. Corrupt the target columns
    corrupter = DataCorrupter(
        mechanism="MCAR",
        corruption_fraction=fraction,
        numeric_columns=target_cols,
        random_state=random_state,
    )
    corrupted, mask, original = corrupter.corrupt(df_clean)

    # 2. Impute with TimeSeriesImputer
    imputer = TimeSeriesImputer(
        time_column=time_col,
        lags=[1, 2, 3, 5],
        rolling_windows=[10, 30],
        initial_strategy="forward_fill",
        max_iter=10,
        random_state=random_state,
        verbose=0,
    )
    imputed = imputer.fit_transform(corrupted)

    # 3. Baselines: forward-fill and mean imputation
    corrupted_ffill = corrupted.copy()
    corrupted_mean = corrupted.copy()
    for col in target_cols:
        corrupted_ffill[col] = corrupted_ffill[col].ffill().bfill()
        corrupted_mean[col] = corrupted_mean[col].fillna(original[col].mean())

    # 4. Evaluate per column
    results = {"fraction": fraction}

    for col in target_cols:
        m = mask[col]
        if m.sum() == 0:
            results[f"{col}_rmse"] = np.nan
            results[f"{col}_mae"] = np.nan
            continue

        true_vals = original.loc[m, col].to_numpy(dtype=float)
        imp_vals  = imputed.loc[m, col].to_numpy(dtype=float)
        ff_vals   = corrupted_ffill.loc[m, col].to_numpy(dtype=float)
        mean_vals = corrupted_mean.loc[m, col].to_numpy(dtype=float)

        rmse_imp  = np.sqrt(mean_squared_error(true_vals, imp_vals))
        rmse_ff   = np.sqrt(mean_squared_error(true_vals, ff_vals))
        rmse_mean = np.sqrt(mean_squared_error(true_vals, mean_vals))
        mae_imp   = np.mean(np.abs(true_vals - imp_vals))
        mae_ff    = np.mean(np.abs(true_vals - ff_vals))
        mae_mean  = np.mean(np.abs(true_vals - mean_vals))

        results[f"{col}_rmse"]         = round(rmse_imp, 2)
        results[f"{col}_rmse_ffill"]   = round(rmse_ff, 2)
        results[f"{col}_rmse_mean"]    = round(rmse_mean, 2)
        results[f"{col}_mae"]          = round(mae_imp, 2)
        results[f"{col}_mae_ffill"]    = round(mae_ff, 2)
        results[f"{col}_mae_mean"]     = round(mae_mean, 2)

    results["n_corrupted_total"] = int(mask[target_cols].sum().sum())
    return results

print("✅ TS benchmark function defined")


✅ TS benchmark function defined


In [20]:
# ---------------------------------------------------------------------------
# 12. Run all 3 scenarios: 1%, 10%, 30%
# ---------------------------------------------------------------------------

FRACTIONS = [0.01, 0.10, 0.30]
SEED = 42

all_ts_results = []
for frac in FRACTIONS:
    print(f"Running {frac:.0%} corruption...", end=" ")
    row = run_ts_benchmark(
        df_gpu,              # the clean GPU DataFrame (time_sec + TARGET_COLS)
        fraction=frac,
        target_cols=TARGET_COLS,
        random_state=SEED,
    )
    all_ts_results.append(row)
    n_c = row["n_corrupted_total"]
    print(f"({n_c} cells corrupted)")

# Build results DataFrame
results_ts_df = pd.DataFrame(all_ts_results).set_index("fraction")
results_ts_df.index = results_ts_df.index.map(lambda x: f"{x:.0%}")
results_ts_df.index.name = "Corruption Level"

print("\n✅ All TS scenarios complete\n")
results_ts_df


Running 1% corruption... (4 cells corrupted)
Running 10% corruption... 

c:\repositories\mixed-imputer\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


(82 cells corrupted)
Running 30% corruption... (226 cells corrupted)

✅ All TS scenarios complete



c:\repositories\mixed-imputer\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,GPU Temperature [°C]_rmse,GPU Temperature [°C]_rmse_ffill,GPU Temperature [°C]_rmse_mean,GPU Temperature [°C]_mae,GPU Temperature [°C]_mae_ffill,GPU Temperature [°C]_mae_mean,Fan Speed (RPM) [RPM]_rmse,Fan Speed (RPM) [RPM]_mae,n_corrupted_total,Fan Speed (RPM) [RPM]_rmse_ffill,Fan Speed (RPM) [RPM]_rmse_mean,Fan Speed (RPM) [RPM]_mae_ffill,Fan Speed (RPM) [RPM]_mae_mean
Corruption Level,,,,,,,,,,,,,
1%,0.49,0.87,6.26,0.42,0.75,5.27,NaN,NaN,4,NaN,NaN,NaN,NaN
10%,1.20,1.33,5.83,0.64,0.60,4.50,213.20,42.50,82,182.09,378.89,43.67,294.78
30%,1.21,1.32,5.42,0.76,0.68,4.21,118.41,26.05,226,114.85,324.05,35.52,273.52


## 13. Visualizations

### 13a. RMSE — TimeSeriesImputer vs. Baselines

Lower is better. Dashed lines show forward-fill and mean-imputation baselines.

In [21]:
# ── RMSE bar chart: TimeSeriesImputer vs. baselines ──
import plotly.graph_objects as go

rmse_imp_cols = [c for c in results_ts_df.columns if c.endswith("_rmse") and "_ffill" not in c and "_mean" not in c]
rmse_imp_data = results_ts_df[rmse_imp_cols].copy()
rmse_imp_data.columns = [c.replace("_rmse", "") for c in rmse_imp_cols]

COLORS_TS = ["#66c2a5", "#fc8d62", "#8da0cb"]  # green, orange, purple

fig_ts_rmse = go.Figure()

# TimeSeriesImputer bars
for i, col in enumerate(rmse_imp_data.columns):
    fig_ts_rmse.add_trace(go.Bar(
        name=f"{col} (TimeSeriesImputer)",
        x=rmse_imp_data.index,
        y=rmse_imp_data[col],
        marker_color=COLORS_TS[0],
        text=rmse_imp_data[col].round(1),
        textposition="outside",
    ))

# Forward-fill baseline (dashed scatter)
ff_cols = [c for c in results_ts_df.columns if c.endswith("_rmse_ffill")]
ff_data = results_ts_df[ff_cols].copy()
ff_data.columns = [c.replace("_rmse_ffill", "") for c in ff_cols]

for i, col in enumerate(ff_data.columns):
    fig_ts_rmse.add_trace(go.Scatter(
        name=f"{col} (Forward-fill baseline)",
        x=ff_data.index,
        y=ff_data[col],
        mode="lines+markers+text",
        marker=dict(color=COLORS_TS[1], size=10, symbol="diamond"),
        line=dict(dash="dash", color=COLORS_TS[1]),
        text=ff_data[col].round(1),
        textposition="top center",
    ))

# Mean-imputation baseline (dotted scatter)
mean_cols = [c for c in results_ts_df.columns if c.endswith("_rmse_mean")]
mean_data = results_ts_df[mean_cols].copy()
mean_data.columns = [c.replace("_rmse_mean", "") for c in mean_cols]

for i, col in enumerate(mean_data.columns):
    fig_ts_rmse.add_trace(go.Scatter(
        name=f"{col} (Mean baseline)",
        x=mean_data.index,
        y=mean_data[col],
        mode="lines+markers+text",
        marker=dict(color=COLORS_TS[2], size=10, symbol="cross"),
        line=dict(dash="dot", color=COLORS_TS[2]),
        text=mean_data[col].round(1),
        textposition="bottom center",
    ))

fig_ts_rmse.update_layout(
    title="GPU Sensor Log — RMSE: TimeSeriesImputer vs. Baselines",
    xaxis_title="Corruption Level",
    yaxis_title="RMSE (lower is better)",
    barmode="group",
    template="plotly_white",
    height=500,
)
fig_ts_rmse.show()
